In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import uniform, randint
from xgboost import XGBClassifier

In [2]:
# Load Data
data = pd.read_csv("../data/preprocessed_data.csv")
test = pd.read_csv("../data/preprocessed_test.csv")

In [3]:

# Selected features and target
selected_columns = [
    "CryoSleep", "Age", "VIP", "RoomService", "FoodCourt",
    "ShoppingMall", "Spa", "VRDeck", "Cabin_num", "Side",
    "CabinMidFlag", "HomePlanetEarth", "HomePlanetEuropa",
    "HomePlanetMars", "Destination55 Cancri e",
    "DestinationPSO J318.5-22", "DestinationTRAPPIST-1e",
    "Group1", "Group2", "Group3", "Group4", "Group5",
    "Group6", "Group7", "DeckA", "DeckB", "DeckC",
    "DeckD", "DeckE", "DeckF", "DeckG",
    "FamilyMembersCabinCount", "avg_spending_family"
]

target_column = "Transported"

X = data[selected_columns]
y = data[target_column]

# Split into train and holdout
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

# Optional: scale numeric features (not strictly needed for XGBoost)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)


In [4]:

# Define parameter grid
param_dist = {
    'n_estimators': randint(300, 1000),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5)
}

xgb_model = XGBClassifier(
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Randomized search
random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist,
    n_iter=50,          # try 50 combinations
    scoring='f1',       # optimize for F1 score (balance FP/FN)
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

# Best parameters
print("Best params:", random_search.best_params_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'colsample_bytree': np.float64(0.7572390898667042), 'gamma': np.float64(4.460232775885567), 'learning_rate': np.float64(0.1362277251994526), 'max_depth': 4, 'n_estimators': 576, 'subsample': np.float64(0.8010548372420768)}


d:\SBU DS\Spring 2026\AMS 580 - Statistical Learning\Spaceship titanic project\spaceship-titanic\env\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:28:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [5]:
best_xgb = random_search.best_estimator_
y_pred = best_xgb.predict(X_holdout_scaled)

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_holdout, y_pred))
print(confusion_matrix(y_holdout, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.81      0.81       432
           1       0.81      0.83      0.82       438

    accuracy                           0.82       870
   macro avg       0.82      0.82      0.82       870
weighted avg       0.82      0.82      0.82       870

[[348  84]
 [ 74 364]]


In [6]:
import pandas as pd

# Convert to DataFrame (keep same column names if you have them)
holdout_df = pd.DataFrame(X_holdout_scaled, columns=X_holdout.columns)

# Add Actual and Pred columns
holdout_df["Actual"] = y_holdout
holdout_df["Pred"] = y_pred

# False positives and negatives
false_positive = holdout_df[(holdout_df["Pred"] == 1) & (holdout_df["Actual"] == 0)]
false_negative = holdout_df[(holdout_df["Pred"] == 0) & (holdout_df["Actual"] == 1)]

false_positive.describe(), false_negative.describe()

(       CryoSleep        Age        VIP  RoomService  FoodCourt  ShoppingMall  \
 count  22.000000  22.000000  22.000000    22.000000  22.000000     22.000000   
 mean    0.677894   0.272305   0.149704    -0.219146  -0.171112      0.213678   
 std     0.995010   1.226981   1.421981     0.542883   0.270653      1.555169   
 min    -0.745163  -1.234892  -0.153463    -0.335393  -0.281427     -0.304376   
 25%    -0.745163  -0.642173  -0.153463    -0.335393  -0.281427     -0.304376   
 50%     1.341988  -0.031549  -0.153463    -0.335393  -0.281427     -0.304376   
 75%     1.341988   0.839623  -0.153463    -0.335393  -0.281427     -0.304376   
 max     1.341988   2.809541   6.516219     2.211436   0.782640      6.326621   
 
              Spa     VRDeck  Cabin_num       Side  ...      DeckB  \
 count  22.000000  22.000000  22.000000  22.000000  ...  22.000000   
 mean   -0.262382  -0.262987   0.445403  -0.188597  ...   0.485793   
 std     0.023241   0.010683   1.228588   1.006496  ...   1

# Submission

In [7]:
test_selected = test[selected_columns]
test_selected_scaled = scaler.transform(test_selected)

In [8]:
test_preds = best_xgb.predict(test_selected_scaled)
test_preds = ["True" if pred == 1 else "False" for pred in test_preds]

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_preds
})

In [9]:
submission.to_csv("../output/xgboost/submission.csv", index=False)